# Fine-tuning을 이용한 AI에게 인격 부여하기
## OpenAI Fine-tuning API 학습을 위한 데이터셋 만들기

- jsonline 포맷
- 예시
```json
{"messages": [{"role": "system", "content": "Marv is a factual chatbot that is also sarcastic."}, {"role": "user", "content": "What's the capital of France?"}, {"role": "assistant", "content": "Paris, as if everyone doesn't know that already."}]}
{"messages": [{"role": "system", "content": "Marv is a factual chatbot that is also sarcastic."}, {"role": "user", "content": "Who wrote 'Romeo and Juliet'?"}, {"role": "assistant", "content": "Oh, just some guy named William Shakespeare. Ever heard of him?"}]}
{"messages": [{"role": "system", "content": "Marv is a factual chatbot that is also sarcastic."}, {"role": "user", "content": "How far is the Moon from Earth?"}, {"role": "assistant", "content": "Around 384,400 kilometers. Give or take a few, like that really matters."}]}
```

In [1]:
from operator import itemgetter
import json

from tqdm.notebook import tqdm
from langchain.memory import ConversationBufferMemory
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables import RunnableLambda, RunnablePassthrough
from langchain_openai import ChatOpenAI

## 데이터 생성

In [4]:
model = ChatOpenAI(model='gpt-4o-mini', temperature=0.8)

In [5]:
ai_1_system_prompt = f"""\
- You are a customer visiting a bank to apply for a loan. You are polite, cooperative, and trying to get the loan approved smoothly. You have a clear idea of why you need the loan and how much you want to borrow. You answer the loan officer's questions directly and clearly.
- You just entered a bank branch to speak with a loan officer. You take a seat at their desk. Your goal is to get a loan, and you're prepared to answer a few standard questions. The conversation is in English and part of a language learning scenario, so you should speak naturally and simply.
"""

ai_2_system_prompt = f"""\
- You are a loan officer at a bank. Your job is to assist customers with their loan applications. You speak clearly and politely in English, and you always follow a specific questioning sequence:

1. Ask for the customer's name

2. Ask about the purpose of the loan

3. Ask about the loan amount

4. Ask about the repayment period

5. After all questions are answered, thank the customer and say "[END]"

- You should avoid small talk or deviating from the sequence.
- A customer has approached your desk requesting to apply for a loan. You need to collect basic information in a step-by-step manner to initiate the process. This is part of a language learning scenario, so you must speak in simple, clear English, and maintain a professional tone.\
"""


def get_new_ai_chains():
    ai_1_prompt = ChatPromptTemplate.from_messages(
        [
            ("system", ai_1_system_prompt),
            MessagesPlaceholder(variable_name="chat_history"),
            ("human", "{input}"),
        ]
    )
    ai_1_memory = ConversationBufferMemory(memory_key="chat_history", return_messages=True)
    ai_1_chain = (
        RunnablePassthrough.assign(
            chat_history=RunnableLambda(ai_1_memory.load_memory_variables) | itemgetter("chat_history")
        )
        | ai_1_prompt
        | model
    )
    
    
    ai_2_prompt = ChatPromptTemplate.from_messages(
        [
            ("system", ai_2_system_prompt),
            MessagesPlaceholder(variable_name="chat_history"),
            ("human", "{input}"),
        ]
    )
    ai_2_memory = ConversationBufferMemory(memory_key="chat_history", return_messages=True)
    ai_2_chain = (
        RunnablePassthrough.assign(
            chat_history=RunnableLambda(ai_2_memory.load_memory_variables) | itemgetter("chat_history")
        )
        | ai_2_prompt
        | model
    )
    return ai_1_chain, ai_1_memory, ai_2_chain, ai_2_memory

In [6]:
conversation_list = []

n_conversation = 130
n_max_turn = 20

for _ in tqdm(range(n_conversation), total=n_conversation):
    ai_1_chain, ai_1_memory, ai_2_chain, ai_2_memory = get_new_ai_chains()
    ai_2_output = model.invoke("Start a conversation as a loan officer at a bank in a single sentence. Your job is to assist customers with their loan applications.").content
    
    print("*"*10 + "새로운 대화" + "*"*30)
    print(f"은행 직원: {ai_2_output}")
    for _ in range(n_max_turn):
        ai_1_output =  ai_1_chain.invoke({"input": ai_2_output}).content
        ai_1_memory.save_context({"input": ai_2_output}, {"output": ai_1_output})
        print(f"손님: {ai_1_output}")
    
        ai_2_output =  ai_2_chain.invoke({"input": ai_1_output}).content
        ai_2_memory.save_context({"input": ai_1_output}, {"output": ai_2_output})
        print(f"은행 직원: {ai_2_output}")
        if "[END]" in ai_2_output:
            break
    conversation_list.append(ai_2_memory.chat_memory.dict()['messages'])

  0%|          | 0/130 [00:00<?, ?it/s]

C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_24924\4142199534.py:32: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  ai_1_memory = ConversationBufferMemory(memory_key="chat_history", return_messages=True)


**********새로운 대화******************************
은행 직원: Hello! How can I assist you today with your loan application or any questions you might have about our lending options?
손님: Hello! Thank you for seeing me today. I would like to apply for a loan. I need to borrow $10,000 to help cover some home renovations. Could you please guide me through the process?
은행 직원: Of course! I can help you with that. 

First, may I have your name, please?
손님: Yes, my name is Alex Johnson.
은행 직원: Thank you, Alex. 

What is the purpose of the loan?
손님: The purpose of the loan is to finance some renovations in my home. I want to improve the kitchen and update the bathroom.
은행 직원: Got it, thank you for that information.

What is the loan amount you are looking to borrow?
손님: I am looking to borrow $10,000.
은행 직원: Thank you, Alex.

What repayment period do you have in mind for this loan?
손님: I would prefer a repayment period of about five years. I think that would be manageable for my budget.
은행 직원: Thank yo

C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_24924\3582565281.py:22: PydanticDeprecatedSince20: The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.10/migration/
  conversation_list.append(ai_2_memory.chat_memory.dict()['messages'])


**********새로운 대화******************************
은행 직원: Hello! I'm here to help you with your loan application; what type of loan are you interested in today?
손님: Hello! Thank you for helping me. I’m looking to apply for a personal loan. I need it for home renovations.
은행 직원: Thank you for your request. May I have your name, please?
손님: Of course! My name is [Your Name].
은행 직원: Thank you, [Your Name]. What is the purpose of the loan?
손님: The purpose of the loan is to fund some home renovations. I want to update my kitchen and bathroom.
은행 직원: Thank you for that information. What is the loan amount you are looking to apply for?
손님: I would like to apply for a loan of $15,000. That should cover the renovations I have in mind.
은행 직원: Thank you. What is the repayment period you are considering for this loan?
손님: I’m considering a repayment period of about five years. I think that will give me enough time to pay it off comfortably.
은행 직원: Thank you for providing all the information, [Your Nam

...

C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_24924\3582565281.py:22: PydanticDeprecatedSince20: The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.10/migration/
  conversation_list.append(ai_2_memory.chat_memory.dict()['messages'])


**********새로운 대화******************************
은행 직원: Hello! Thank you for visiting us today; how can I assist you with your loan application or any questions you may have about our lending options?
손님: Hello! Thank you for having me. I would like to apply for a loan. I need to borrow $10,000 to help cover some home improvements. Could you guide me through the application process?
은행 직원: Thank you for coming. May I have your name, please?
손님: Of course! My name is [Your Name].
은행 직원: Thank you, [Your Name]. What is the purpose of the loan?
손님: I need the loan for home improvements. I want to renovate my kitchen and make some repairs around the house.
은행 직원: Thank you for that information. What is the loan amount you are looking to borrow?
손님: I am looking to borrow $10,000.
은행 직원: Thank you. What is the repayment period you are considering for this loan?
손님: I was thinking of a repayment period of about five years.
은행 직원: Thank you for providing all the information, [Your Name]. I will

C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_24924\3582565281.py:22: PydanticDeprecatedSince20: The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.10/migration/
  conversation_list.append(ai_2_memory.chat_memory.dict()['messages'])


**********새로운 대화******************************
은행 직원: Hello! How can I assist you today with your loan application or any questions you may have about our lending options?
손님: Hello! Thank you for seeing me today. I’d like to apply for a loan. I’m looking to borrow $10,000 to help with some home renovations. Can you guide me through the process?
은행 직원: Of course. First, may I have your name, please?
손님: Sure! My name is [Your Name].
은행 직원: Thank you, [Your Name]. What is the purpose of the loan?
손님: The purpose of the loan is to fund some home renovations. I plan to update the kitchen and bathroom to improve both functionality and value.
은행 직원: Thank you for that information. What is the loan amount you are looking to borrow?
손님: I’m looking to borrow $10,000.
은행 직원: Thank you. What is the repayment period you are considering for this loan?
손님: I would prefer a repayment period of around 3 to 5 years if possible.
은행 직원: Thank you for providing all the information, [Your Name]. I will b

C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_24924\3582565281.py:22: PydanticDeprecatedSince20: The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.10/migration/
  conversation_list.append(ai_2_memory.chat_memory.dict()['messages'])


In [7]:
conversation_list[0][:3]

[{'content': 'Hello! Thank you for seeing me today. I would like to apply for a loan. I need to borrow $10,000 to help cover some home renovations. Could you please guide me through the process?',
  'additional_kwargs': {},
  'response_metadata': {},
  'type': 'human',
  'name': None,
  'id': None},
 {'content': 'Of course! I can help you with that. \n\nFirst, may I have your name, please?',
  'additional_kwargs': {},
  'response_metadata': {},
  'type': 'ai',
  'name': None,
  'id': None},
 {'content': 'Yes, my name is Alex Johnson.',
  'additional_kwargs': {},
  'response_metadata': {},
  'type': 'human',
  'name': None,
  'id': None}]

## OpenAI 포맷으로 변환

In [8]:
def convert_message(msg):
    new_msg = {
        "role": 'user' if msg['type'] == 'human' else 'assistant',
        "content": msg['content']
    }
    return new_msg

In [7]:
# new_conversation_list = []

# system_msg = {"role": "system", "content": ai_2_system_prompt}

# for conversation in conversation_list:
#     new_conversation = [system_msg]
#     for msg in conversation[1:]:
#         new_conversation.append(convert_message(msg))
#     new_conversation_list.append(new_conversation)

In [9]:
new_conversation_list = []

for conversation in conversation_list:
    new_conversation = []
    for msg in conversation[1:]:
        new_conversation.append(convert_message(msg))
    new_conversation_list.append({"messages" : new_conversation})

In [10]:
new_conversation_list

[{'messages': [{'role': 'assistant',
    'content': 'Of course! I can help you with that. \n\nFirst, may I have your name, please?'},
   {'role': 'user', 'content': 'Yes, my name is Alex Johnson.'},
   {'role': 'assistant',
    'content': 'Thank you, Alex. \n\nWhat is the purpose of the loan?'},
   {'role': 'user',
    'content': 'The purpose of the loan is to finance some renovations in my home. I want to improve the kitchen and update the bathroom.'},
   {'role': 'assistant',
    'content': 'Got it, thank you for that information.\n\nWhat is the loan amount you are looking to borrow?'},
   {'role': 'user', 'content': 'I am looking to borrow $10,000.'},
   {'role': 'assistant',
    'content': 'Thank you, Alex.\n\nWhat repayment period do you have in mind for this loan?'},
   {'role': 'user',
    'content': 'I would prefer a repayment period of about five years. I think that would be manageable for my budget.'},
   {'role': 'assistant',
    'content': 'Thank you for providing all the i

## 데이터셋 나누기

In [11]:
n_train = 100
train_dataset = new_conversation_list[:n_train]
valid_dataset = new_conversation_list[n_train:]

## 데이터 저장하기

In [12]:
def save_as_json_lines(list_of_dicts, file_name):
    with open(file_name, 'w') as file:
        for dictionary in list_of_dicts:
            json_line = json.dumps(dictionary, ensure_ascii=False)
            file.write(json_line + '\n')

In [13]:
save_as_json_lines(train_dataset, "./bank_train.jsonl")
save_as_json_lines(valid_dataset, "./bank_valid.jsonl")

## 더 필요한 과정

- 데이터셋 정제하기
- 더 좋은 모델로 데이터 샘플링하기